# Imagined Speech EEG Dataset Triage

Goal: choose datasets for a first imagined/covert speech EEG reconstruction
pipeline. The practical target is **imagined EEG -> prompt label/audio
embedding retrieval**, not direct waveform generation on day one.

This notebook consumes the lightweight remote probes generated by:

```bash
python3 scripts/probe_eeg_audio_datasets.py \
  --only feis_3554128 kara_one ugr_mindvoice ds003626 ds004306 ds005170 ds007591 ds004408 \
  --artifact-dir outputs/imagined_speech_probe_artifacts \
  --json-out outputs/imagined_speech_dataset_probe_results.json \
  --md-out outputs/imagined_speech_dataset_probe_results.md \
  --detail-md-out outputs/imagined_speech_dataset_probe_details.md
```

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
PROBE_JSON = ROOT / "outputs" / "imagined_speech_dataset_probe_results.json"
STRATEGY_JSON = ROOT / "outputs" / "imagined_speech_dataset_strategy.json"
INDEX_TEMPLATE_CSV = ROOT / "outputs" / "imagined_speech_unified_index_template.csv"
ARTIFACT_DIR = ROOT / "outputs" / "imagined_speech_probe_artifacts"

assert STRATEGY_JSON.exists(), STRATEGY_JSON
strategy = pd.DataFrame(json.loads(STRATEGY_JSON.read_text()))
strategy

## Probe Status

The probe only verifies public availability and small metadata/audio/event
snippets. It intentionally does not pull full EEG archives.

In [ ]:
probe_results = json.loads(PROBE_JSON.read_text()) if PROBE_JSON.exists() else []

def probe_status(row):
    if "error" in row:
        return "ERROR", row["error"]
    errors = [t for t in row.get("targets", []) if "error" in t]
    status = "PARTIAL" if errors else "OK"
    evidence = []
    if row.get("file_count") is not None:
        evidence.append(f"zenodo_files={row['file_count']}")
    for target in row.get("targets", [])[:6]:
        if "error" in target:
            evidence.append(f"{target.get('label')}: {target.get('error')}")
        else:
            parsed = target.get("parsed", {})
            if isinstance(parsed, list):
                evidence.append(f"{target['label']}: {len(parsed)} items")
            elif isinstance(parsed, dict):
                if target.get("kind") in {"tsv", "csv"}:
                    evidence.append(f"{target['label']}: {len(parsed.get('columns', []))} cols")
                elif target.get("kind") == "wav":
                    evidence.append(f"{target['label']}: {parsed.get('sample_rate')} Hz")
                elif target.get("kind") == "json":
                    evidence.append(f"{target['label']}: json")
                else:
                    evidence.append(f"{target['label']}: {target.get('bytes_read')} bytes")
    return status, "; ".join(evidence)

rows = []
for row in probe_results:
    status, evidence = probe_status(row)
    rows.append({
        "dataset": row.get("dataset_id"),
        "source": row.get("source"),
        "priority": row.get("priority"),
        "status": status,
        "evidence": evidence,
    })
probe_table = pd.DataFrame(rows)
probe_table

## Decision Matrix

Primary training data should have an imagined/covert condition and an
explicit time window. Heard speech datasets are still useful, but only
for acoustic or phoneme pretraining.

In [ ]:
display(strategy[[
    "rank",
    "dataset",
    "role",
    "language",
    "imagined_window",
    "target_proxy",
    "overt_audio",
    "recommendation",
]])

## Artifact Inventory

This is a quick way to see what the probe saved locally. Binary and
audio-like probe artifacts are ignored by git.

In [ ]:
artifact_rows = []
if ARTIFACT_DIR.exists():
    for path in sorted(ARTIFACT_DIR.rglob("*")):
        if path.is_file():
            artifact_rows.append({
                "dataset": path.parent.name,
                "file": str(path.relative_to(ROOT)),
                "bytes": path.stat().st_size,
            })
pd.DataFrame(artifact_rows)

## Unified Trial Index Schema

Every downloaded dataset should be converted into this narrow table.
The table is allowed to start with labels/proxy targets; direct waveform
targets should only be used when the dataset provides overt/prompt audio.

In [ ]:
index_template = pd.read_csv(INDEX_TEMPLATE_CSV)
index_template

In [ ]:
REQUIRED_COLUMNS = [
    "dataset",
    "subject",
    "trial",
    "mode",
    "label",
    "eeg_start",
    "eeg_end",
    "audio_path_or_label",
    "target_kind",
    "source_path",
    "notes",
]
missing = [col for col in REQUIRED_COLUMNS if col not in index_template.columns]
assert not missing, missing
print("Unified index columns are ready:", REQUIRED_COLUMNS)

## BIDS Event Helper

Use this for UGR-MINDVOICE or any future BIDS-style imagined/covert
speech dataset. It converts an events file into the same index schema.

In [ ]:
def bids_events_to_unified_index(
    events_path: Path,
    dataset: str,
    subject: str,
    mode_default: str = "covert",
    label_col: str = "trial_type",
    mode_col: str | None = None,
) -> pd.DataFrame:
    events = pd.read_csv(events_path, sep="\t", encoding="utf-8-sig")
    rows = []
    for trial, event in events.reset_index(drop=True).iterrows():
        onset = event.get("onset")
        duration = event.get("duration", 0)
        label = event.get(label_col, "")
        mode = event.get(mode_col, mode_default) if mode_col else mode_default
        eeg_end = ""
        try:
            eeg_end = float(onset) + float(duration)
        except Exception:
            pass
        rows.append({
            "dataset": dataset,
            "subject": subject,
            "trial": trial,
            "mode": mode,
            "label": label,
            "eeg_start": onset,
            "eeg_end": eeg_end,
            "audio_path_or_label": label,
            "target_kind": "event_label",
            "source_path": str(events_path),
            "notes": "Generated from BIDS events; confirm task-specific label columns before training.",
        })
    return pd.DataFrame(rows)

possible_ugr_events = sorted(ARTIFACT_DIR.glob("ugr_mindvoice/*events*.tsv"))
if possible_ugr_events:
    display(bids_events_to_unified_index(possible_ugr_events[0], "ugr_mindvoice", "sub-05").head())
else:
    print("No UGR events artifact found yet. Re-run the probe after network access is available.")

## Recommended Next Execution

1. Extract one FEIS subject (`experiments/01/thinking.zip`,
   `stimuli.zip`, `speaking.zip`, and `wavs/01/wavs/*.wav`).
2. Fill a real per-trial index for FEIS using the schema above.
3. Train a retrieval baseline: imagined EEG epoch -> prompt audio
   embedding or label.
4. Use DS004408 only as heard-speech pretraining, then test transfer to
   imagined FEIS/KARA/UGR windows.